### Load Data

In [252]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('titanic')

print("Path to competition files:", path)

Path to competition files: C:\Users\robmc\.cache\kagglehub\competitions\titanic


In [253]:
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

train_data = pd.read_csv(path + '/train.csv')
test_data = pd.read_csv(path + '/test.csv')

### Tutorial - Baseline Score

In [170]:
from sklearn.ensemble import RandomForestClassifier

y = train_data["Survived"]

features = ["Pclass", "Sex", "SibSp", "Parch"]
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


# Preparing the Data

In [254]:
train = train_data.copy()
test = test_data.copy()

train.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [255]:
train["Title"] = train["Name"].str.extract(' ([A-Za-z]+)\\.', expand=False)

train = train.drop(columns=['Name', 'Ticket'])
train.head(1)

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,Title
0,1,0,3,male,22.0,1,0,7.25,NaN,S,Mr


In [256]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    str    
 4   Age          714 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Fare         891 non-null    float64
 8   Cabin        204 non-null    str    
 9   Embarked     889 non-null    str    
 10  Title        891 non-null    str    
dtypes: float64(2), int64(5), str(4)
memory usage: 76.7 KB


In [257]:
train['Embarked'] = train['Embarked'].fillna('S')

train['Age'] = train['Age'].fillna(train['Age'].median())

train["Deck"] = ""
for index, cabin in train['Cabin'].items():
    train.loc[index, "Deck"] = "U" if pd.isna(cabin) else cabin[0]

train = train.drop(columns=['Cabin'])

In [258]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    str    
 4   Age          891 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Fare         891 non-null    float64
 8   Embarked     891 non-null    str    
 9   Title        891 non-null    str    
 10  Deck         891 non-null    str    
dtypes: float64(2), int64(5), str(4)
memory usage: 76.7 KB


In [259]:
train['FamSize'] = train['SibSp'] + train['Parch'] + 1
train = train.drop(columns=['SibSp', 'Parch'])

In [260]:
train['Sex'] = train['Sex'].map({'female': 0, 'male': 1})

In [261]:
train['Embarked'].unique()

<StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str

In [262]:
train = pd.get_dummies(train, columns=['Embarked'])

In [263]:
train['Title'].value_counts()

Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64

In [264]:
train['Title'] = train['Title'].replace(['Ms', 'Mlle'], 'Miss')
train['Title'] = train['Title'].replace(['Sir'], 'Mr')
train['Title'] = train['Title'].replace(['Lady', 'Mme'], 'Mrs')
train['Title'] = train['Title'].replace(['Jonkheer', 'Countess', 'Don', 'Dona', 'Rev', 'Capt', 'Col', 'Major', 'Dr'], 'Important')

In [265]:
train['Title'].value_counts()

Title
Mr           518
Miss         185
Mrs          127
Master        40
Important     21
Name: count, dtype: int64

In [266]:
train = pd.get_dummies(train, columns=['Title'])

In [267]:
train['Deck'].value_counts()

Deck
U    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

In [268]:
train['Deck'] = train['Deck'].map({'G': 0, 'U': 1, 'F': 2, 'E': 3, 'D': 4, 'C': 5, 'B': 6, 'A': 7, 'T': 8})

In [269]:
train.head(2)

,PassengerId,Survived,Pclass,Sex,Age,Fare,Deck,FamSize,Embarked_C,Embarked_Q,Embarked_S,Title_Important,Title_Master,Title_Miss,Title_Mr,Title_Mrs
0,1,0,3,1,22.0,7.2500,1,2,False,False,True,False,False,False,True,False
1,2,1,1,0,38.0,71.2833,5,2,True,False,False,False,False,False,False,True


In [270]:
test["Title"] = test["Name"].str.extract(' ([A-Za-z]+)\\.', expand=False)

test = test.drop(columns=['Name', 'Ticket'])

test['Embarked'] = test['Embarked'].fillna('S')

test['Age'] = test['Age'].fillna(test['Age'].median())

test["Deck"] = ""
for index, cabin in test['Cabin'].items():
    test.loc[index, "Deck"] = "U" if pd.isna(cabin) else cabin[0]

test = test.drop(columns=['Cabin'])

test['FamSize'] = test['SibSp'] + test['Parch'] + 1
test = test.drop(columns=['SibSp', 'Parch'])

test['Sex'] = test['Sex'].map({'female': 0, 'male': 1})

test = pd.get_dummies(test, columns=['Embarked'])

test['Title'] = test['Title'].replace(['Ms', 'Mlle'], 'Miss')
test['Title'] = test['Title'].replace(['Sir'], 'Mr')
test['Title'] = test['Title'].replace(['Lady', 'Mme'], 'Mrs')
test['Title'] = test['Title'].replace(['Jonkheer', 'Countess', 'Don', 'Dona', 'Rev', 'Capt', 'Col', 'Major', 'Dr'], 'Important')
test = pd.get_dummies(test, columns=['Title'])

test['Deck'] = test['Deck'].map({'G': 0, 'U': 1, 'F': 2, 'E': 3, 'D': 4, 'C': 5, 'B': 6, 'A': 7, 'T': 8})

In [271]:
print(test.isna().sum())

PassengerId        0
Pclass             0
Sex                0
Age                0
Fare               1
Deck               0
FamSize            0
Embarked_C         0
Embarked_Q         0
Embarked_S         0
Title_Important    0
Title_Master       0
Title_Miss         0
Title_Mr           0
Title_Mrs          0
dtype: int64


In [272]:
fare_median = train["Fare"].median()
test["Fare"] = test["Fare"].fillna(fare_median)

In [273]:
print(test.isna().sum())

PassengerId        0
Pclass             0
Sex                0
Age                0
Fare               0
Deck               0
FamSize            0
Embarked_C         0
Embarked_Q         0
Embarked_S         0
Title_Important    0
Title_Master       0
Title_Miss         0
Title_Mr           0
Title_Mrs          0
dtype: int64


# Base Decision Tree

In [189]:
train_set = train.copy()
test_set = test.copy()

from sklearn.tree import DecisionTreeClassifier

X = train_set.drop(columns=['Survived'])
y = train_set['Survived']

dt_base = DecisionTreeClassifier(random_state=1)
dt_base.fit(X, y)

predictions = dt_base.predict(test_set)
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


# Improved Decision Tree

In [237]:
train_set = train.copy()
test_set = test.copy()

X = train_set.drop(columns=['Survived'])
y = train_set['Survived']

In [238]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

params = {'max_depth': [3, 5, 7, 9, 11, 13, 15],
          'min_samples_split': [2, 5, 10, 20, 30],
          'min_samples_leaf': [1, 2, 4, 6, 8, 10, 15, 20],
          'criterion': ['gini', 'entropy']}

dt_base = DecisionTreeClassifier(random_state=1)

grid_search = GridSearchCV(estimator=dt_base, param_grid=params, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)

In [ ]:
grid_search.fit(X, y)

dt_best = grid_search.best_estimator_

print("Best parameters found: ", grid_search.best_params_)

Fitting 5 folds for each of 560 candidates, totalling 2800 fits
Best parameters found:  {'criterion': 'entropy', 'max_depth': 9, 'min_samples_leaf': 10, 'min_samples_split': 30}


In [240]:
dt_best.fit(X, y)

predictions = dt_best.predict(test_set)
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


# Base Random Forest

In [242]:
train_set = train.copy()
test_set = test.copy()

from sklearn.ensemble import RandomForestClassifier

X = train_set.drop(columns=['Survived'])
y = train_set['Survived']

rf_base = RandomForestClassifier(random_state=1)
rf_base.fit(X, y)

predictions = rf_base.predict(test_set)
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


# Improved Random Forest

In [274]:
train_set = train.copy()
test_set = test.copy()

X = train_set.drop(columns=['Survived'])
y = train_set['Survived']

In [275]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

params = {
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [2, 5, 10],
    'n_estimators': [100, 250, 500],
    'max_features': ['sqrt', 'log2', 1.0],
    'criterion': ['gini', 'entropy'],
    'ccp_alpha': [0.0, 0.02]
}

rf_base = RandomForestClassifier(random_state=1)

grid_search = GridSearchCV(estimator=rf_base, param_grid=params, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)

In [276]:
from contextlib import contextmanager
from joblib import parallel
from sklearn.model_selection import ParameterGrid
from tqdm.auto import tqdm


@contextmanager
def tqdm_joblib(progress_bar):
    class TqdmBatchCompletionCallback(parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            progress_bar.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    original_callback = parallel.BatchCompletionCallBack
    parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield progress_bar
    finally:
        parallel.BatchCompletionCallBack = original_callback
        progress_bar.close()


total_fits = len(list(ParameterGrid(params))) * grid_search.cv
with tqdm_joblib(tqdm(total=total_fits, desc="Random forest grid search")):
    grid_search.fit(X, y)

dt_best = grid_search.best_estimator_

print("Best parameters found: ", grid_search.best_params_)

Random forest grid search:   0%|          | 0/10800 [00:00<?, ?it/s]

Fitting 5 folds for each of 2160 candidates, totalling 10800 fits


Random forest grid search: 100%|██████████| 10800/10800 [06:36<00:00, 27.23it/s]

Best parameters found:  {'ccp_alpha': 0.0, 'criterion': 'gini', 'max_depth': None, 'max_features': 1.0, 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 100}


Best parameters found:  {'ccp_alpha': 0.0, 'criterion': 'gini', 'max_depth': None, 
'max_features': 1.0, 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 100}

In [277]:
dt_best.fit(X, y)

predictions = dt_best.predict(test_set)
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


# Base Gradient Boost

In [278]:
train_set = train.copy()
test_set = test.copy()

X = train_set.drop(columns=['Survived'])
y = train_set['Survived']

In [279]:
from sklearn.ensemble import GradientBoostingClassifier

gbm = GradientBoostingClassifier(random_state=1)

gbm.fit(X, y)

predictions = gbm.predict(test_set)
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


# Improved Gradient Boost

In [280]:
train_set = train.copy()
test_set = test.copy()

X = train_set.drop(columns=['Survived'])
y = train_set['Survived']

In [281]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

gbm = GradientBoostingClassifier(random_state=1)

params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'subsample': [0.8, 1.0],
}

grid_search_gbm = GridSearchCV(estimator=gbm, param_grid=params, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)

In [282]:
grid_search_gbm.fit(X, y)

print("Best Boosting Parameters:", grid_search_gbm.best_params_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best Boosting Parameters: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}


In [283]:
best_gbm = grid_search_gbm.best_estimator_

best_gbm.fit(X, y)

predictions = best_gbm.predict(test_set)
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


# Base XG Boost

In [285]:
train_set = train.copy()
test_set = test.copy()

X = train_set.drop(columns=['Survived'])
y = train_set['Survived']

In [286]:
import xgboost as xgb

xgb = xgb.XGBClassifier(random_state=1)
xgb.fit(X, y)

predictions = xgb.predict(test_set)
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


# Improved XG Boost

In [ ]:
train_set = train.copy()
test_set = test.copy()

X = train_set.drop(columns=['Survived'])
y = train_set['Survived']

# Future Ideas

In [36]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

% of women who survived: 0.7420382165605095
% of men who survived: 0.18890814558058924
